# 2) EDA Completo

## 2.1 Carga de Datos e Inspección general


In [2]:
import pandas as pd

# Cargamos el dataset principal, es decir, el que entrenaremos.
# En nuestro caso, la suma de los dos últimos dígitos de uno de nosotros es 3, por lo que isaremos bank_03.pkl.
try:
    df_entreno = pd.read_pickle("banks/bank_03.pkl")
    print("Datos de entrenamiento cargados correctamente. Dimensiones:", df_entreno.shape)

    # Mostrar las primeras 5 filas para echar un vistazo
    display(df_entreno.head())
except FileNotFoundError:
    print("No se encontró el archivo.")

print("-" * 50)

# Cargamos el dataset de la competición
try:
    df_competicion = pd.read_pickle("banks/bank_competition.pkl")
    print("Datos de la competición cargados correctamente. Dimensiones:", df_competicion.shape)
    display(df_competicion.head())
except FileNotFoundError:
    print("No se encontró el archivo de competición.")

Datos de entrenamiento cargados correctamente. Dimensiones: (11000, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,None,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


--------------------------------------------------
Datos de la competición cargados correctamente. Dimensiones: (162, 16)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
5553,43,management,married,tertiary,no,78,yes,no,cellular,21,nov,36,1,109,1,other
915,34,housemaid,married,secondary,no,0,yes,no,unknown,30,oct,154,1,-1,0,unknown
7652,54,technician,married,secondary,no,3323,yes,yes,cellular,8,apr,59,3,-1,0,unknown
5065,43,blue-collar,single,primary,no,-399,no,yes,cellular,28,jul,662,3,-1,0,unknown
3338,35,blue-collar,married,secondary,no,262,no,no,cellular,15,mar,427,1,181,3,success


## 2.2 Inspección general de datasets (Data Quality)

In [3]:
import pandas as pd
import numpy as np

# Cargamos el archivo del dataset de entrenamiento y obtenemos filas y columnas.
df = pd.read_pickle("banks/bank_03.pkl")

print("1. FORMA DEL DATASET\n")
print(f"Instancias (filas): {df.shape[0]}")
print(f"Variables (columnas): {df.shape[1]}")

print("\n2. TIPO DE VARIABLES\n")
# Separamos variables numéricas y el resto (categóricas/ordinales)
numericas = df.select_dtypes(include=[np.number]).columns.tolist()
categoricas = df.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Numéricas ({len(numericas)}): {numericas}")
print(f"Categóricas/Ordinales ({len(categoricas)}): {categoricas}")

print("\n3. CATEGÓRICAS CON ALTA CARDINALIDAD (>10 valores únicos)\n")
for col in categoricas:
    n_unicos = df[col].nunique()
    if n_unicos > 10:
        print(f"La variable '{col}' tiene {n_unicos} valores únicos.")

print("\n4. VALORES FALTANTES (Nulos, 'unknown' y 'other')\n")

encontrados = False
# Revisamos todas las columnas
for col in df.columns:
    # Contamos los None
    cantidad_nulos = df[col].isna().sum()

    # Contamos cuántos 'unknown' y 'other' hay (solo en texto/categoría)
    cantidad_unknown = 0
    cantidad_other = 0
    if df[col].dtype == 'object' or hasattr(df[col].dtype, 'categories'):
        cantidad_unknown = (df[col] == 'unknown').sum()
        cantidad_other = (df[col] == 'other').sum()

    # Sumamos los tres para saber el total de información faltante en esa columna
    total_faltantes = cantidad_nulos + cantidad_unknown + cantidad_other

    # Si falta algún dato, lo imprimimos con su desglose
    if total_faltantes > 0:
        porcentaje = (total_faltantes / len(df)) * 100

        # Preparamos un texto para saber exactamente qué tipo de faltantes son
        detalles = []
        if cantidad_nulos > 0:
            detalles.append(f"{cantidad_nulos} nulos (None)")
        if cantidad_unknown > 0:
            detalles.append(f"{cantidad_unknown} 'unknown'")
        if cantidad_other > 0:
            detalles.append(f"{cantidad_other} 'other'")

        print(f"La variable '{col}' tiene {total_faltantes} valores faltantes ({porcentaje:.2f}%) -> " + " y ".join(detalles))
        encontrados = True

if not encontrados:
    print("No se encontraron valores faltantes, 'unknown' ni 'other'.")

print("\n5. FILAS DUPLICADAS, COLUMNAS CONSTANTES O DE ID\n")
num_duplicados = df.duplicated().sum()
if num_duplicados > 0:
    print(f"Se han encontrado {num_duplicados} filas exactamente idénticas.")
    print("-> Se deben eliminar para evitar sesgar el modelo y reducir el tamaño del dataset.")
else:
    print("No se encontraron filas duplicadas en el dataset.")
constantes_o_id = False
for col in df.columns:
    if df[col].nunique() == 1:
        print(f"Columna constante encontrada: '{col}'")
        constantes_o_id = True
    elif df[col].nunique() == len(df):
        print(f"Columna tipo ID encontrada (todos los valores distintos): '{col}'")
        constantes_o_id = True
if not constantes_o_id:
    print("No se encontraron columnas constantes ni columnas tipo ID.")

print("\n6. TIPO DE PROBLEMA Y BALANCEO\n")
print("Es un problema de CLASIFICACIÓN (predecir 'yes' o 'no' en deposit).")
print("Distribución de la clase objetivo:")
# normalize=True nos da el porcentaje en vez del recuento
print((df['deposit'].value_counts(normalize=True) * 100).round(2))

1. FORMA DEL DATASET

Instancias (filas): 11000
Variables (columnas): 17

2. TIPO DE VARIABLES

Numéricas (7): ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Categóricas/Ordinales (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'deposit']

3. CATEGÓRICAS CON ALTA CARDINALIDAD (>10 valores únicos)

La variable 'job' tiene 12 valores únicos.
La variable 'month' tiene 12 valores únicos.

4. VALORES FALTANTES (Nulos, 'unknown' y 'other')

La variable 'job' tiene 68 valores faltantes (0.62%) -> 68 'unknown'
La variable 'marital' tiene 402 valores faltantes (3.65%) -> 402 nulos (None)
La variable 'education' tiene 494 valores faltantes (4.49%) -> 494 'unknown'
La variable 'contact' tiene 2309 valores faltantes (20.99%) -> 2309 'unknown'
La variable 'poutcome' tiene 8733 valores faltantes (79.39%) -> 8205 'unknown' y 528 'other'

5. FILAS DUPLICADAS, COLUMNAS CONSTANTES O DE ID

No se encontraron filas duplicadas en e

## 2.3 Análisis Univariante: Distribuciones y Outliers

In [5]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, pointbiserialr

print("ANÁLISIS ESTADÍSTICO DE DISTRIBUCIÓN Y ESCALADO (PARA KNN)")
# Mapeamos el target a numérico para pruebas estadísticas
if 'deposit_bin' not in df.columns:
    df['deposit_bin'] = df['deposit'].map({'yes': 1, 'no': 0})

for col in numericas:
    # 1. Asimetría (Skewness)
    asimetria = df[col].skew()

    # 2. Outliers (Método IQR)
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
    pct_outliers = (outliers / len(df)) * 100

    print(f"\n{col}")

    # Lógica de decisión automatizada para asimetría
    if abs(asimetria) > 1:
        print(f"  - Asimetría ALTA ({asimetria:.2f}). -> Aplicar transformación logarítmica o de Box-Cox para modelos lineales.")
    elif abs(asimetria) > 0.5:
        print(f"  - Asimetría MEDIA ({asimetria:.2f}).")
    else:
        print(f"  - Bastante Simétrica ({asimetria:.2f}).")

    # Lógica de decisión automatizada para escalado (Responde a la sección 4.1 del guion)
    if pct_outliers > 5:
        print(f"  - Outliers: MUCHOS ({pct_outliers:.2f}%). DECISIÓN PARA KNN: Usar RobustScaler.")
    elif pct_outliers > 0:
        print(f"  - Outliers: MODERADOS ({pct_outliers:.2f}%). DECISIÓN PARA KNN: Usar StandardScaler, pero RobustScaler mejor.")
    else:
        print(f"  - Outliers: NINGUNO. DECISIÓN PARA KNN: Usar MinMaxScaler.")




print("\nEXTRA: DECISIÓN DE PREPROCESO AVANZADO PARA 'pdays'")
# Creamos la variable binaria para evaluar matemáticamente su utilidad
df['contactado_previamente'] = (df['pdays'] != -1).astype(int)
contingency_pdays = pd.crosstab(df['contactado_previamente'], df['deposit'])
chi2_pdays, p_pdays, _, _ = chi2_contingency(contingency_pdays)

if p_pdays < 0.05:
    print(f"  - Resultado: Extraer el valor -1 a una nueva variable binaria es ALTAMENTE significativo (p-value = {p_pdays:.2e}).")
    print("  - DECISIÓN PARA PIPELINE: Crear un transformador personalizado que genere esta variable binaria y reemplace los -1 en 'pdays' por la mediana de los contactados o directamente agrupe en bins.")

ANÁLISIS ESTADÍSTICO DE DISTRIBUCIÓN Y ESCALADO (PARA KNN)

age
  - Asimetría MEDIA (0.86).
  - Outliers: MODERADOS (1.55%). DECISIÓN PARA KNN: Usar StandardScaler, pero RobustScaler mejor.

balance
  - Asimetría ALTA (8.25). -> Aplicar transformación logarítmica o de Box-Cox para modelos lineales.
  - Outliers: MUCHOS (9.48%). DECISIÓN PARA KNN: Usar RobustScaler.

day
  - Bastante Simétrica (0.11).
  - Outliers: NINGUNO. DECISIÓN PARA KNN: Usar MinMaxScaler.

duration
  - Asimetría ALTA (2.14). -> Aplicar transformación logarítmica o de Box-Cox para modelos lineales.
  - Outliers: MUCHOS (5.64%). DECISIÓN PARA KNN: Usar RobustScaler.

campaign
  - Asimetría ALTA (5.57). -> Aplicar transformación logarítmica o de Box-Cox para modelos lineales.
  - Outliers: MUCHOS (5.35%). DECISIÓN PARA KNN: Usar RobustScaler.

pdays
  - Asimetría ALTA (2.46). -> Aplicar transformación logarítmica o de Box-Cox para modelos lineales.
  - Outliers: MUCHOS (24.65%). DECISIÓN PARA KNN: Usar RobustScaler.


## 2.4 Casos Especiales (Feature Engineering)

### 2.4.1 El caso de la variable 'pdays'

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

print(" ANÁLISIS PROFUNDO DE LA VARIABLE 'pdays' \n")

# 1. ANÁLISIS DESCRIPTIVO BÁSICO
print("1. DISTRIBUCIÓN Y PESO DEL VALOR ANÓMALO (-1):")
porcentaje_pdays_menos1 = (df['pdays'] == -1).mean() * 100

if porcentaje_pdays_menos1 > 0:
    print(f"  [!] ALERTA: El {porcentaje_pdays_menos1:.2f}% de los clientes tiene valor -1.")
    print("      Esto indica que la inmensa mayoría nunca ha sido contactada antes.")
else:
    print("  [+] No se detectaron valores -1 en la variable pdays.")

print("\n  - Resumen estadístico global de 'pdays':")
desc = df['pdays'].describe()
# Formateamos el describe para que sea visualmente más limpio en consola
print(f"    Media: {desc['mean']:.2f} | Desviación Std: {desc['std']:.2f} | Min: {desc['min']} | Max: {desc['max']}")


# 2. VALIDACIÓN ESTADÍSTICA (Test Chi-Cuadrado)
print("\n2. EVALUACIÓN ESTADÍSTICA DE INGENIERÍA DE CARACTERÍSTICAS:")
# Creamos temporalmente la variable binaria para evaluar su utilidad matemática
df['contactado_previamente'] = (df['pdays'] != -1).astype(int)

# Cruzamos la nueva variable binaria con la decisión de contratar el depósito
contingency_pdays = pd.crosstab(df['contactado_previamente'], df['deposit'])
chi2_pdays, p_pdays, _, _ = chi2_contingency(contingency_pdays)

es_significativo = p_pdays < 0.05
print(f"  - P-value de la nueva variable ('contactado_previamente'): {p_pdays:.2e} -> {'SIGNIFICATIVO\n' if es_significativo else 'NO SIGNIFICATIVO/n'}")


# 3. VEREDICTO FINAL PARA EL PIPELINE
print(" INSTRUCCIONES FINALES PARA EL PIPELINE DE PREPROCESADO\n")

if es_significativo:
    print("[COLUMNA 'pdays']: APLICAR EXTRACCIÓN Y TRANSFORMACIÓN AVANZADA.")
    print(f" -> Motivo: Separar a los clientes 'nunca contactados' (-1) del resto es ALTAMENTE predictivo (p < 0.05).")
    print(" -> Acción para el Pipeline: Crear un CustomTransformer que:")
    print("    1. Genere la nueva variable binaria 'contactado_previamente' (0 si es -1, 1 si no).")
    print("    2. Reemplace el -1 en la columna original de 'pdays' por un valor lógico (ej. la mediana de los sí contactados) para no distorsionar la escala espacial de los modelos numéricos.")
else:
    print("[COLUMNA 'pdays']: MANTENER SIN TRANSFORMACIÓN BINARIA.")
    print(f" -> Motivo: La separación de los no contactados no aporta información extra significativa (p >= 0.05).")
    print(" -> Acción para el Pipeline: Dejar el valor numérico tal cual (aunque el -1 distorsione ligeramente la escala).")

# Limpiamos la columna temporal creada para no ensuciar el dataset principal aún
df = df.drop(columns=['contactado_previamente'])

 ANÁLISIS PROFUNDO DE LA VARIABLE 'pdays' 

1. DISTRIBUCIÓN Y PESO DEL VALOR ANÓMALO (-1):
  [!] ALERTA: El 74.57% de los clientes tiene valor -1.
      Esto indica que la inmensa mayoría nunca ha sido contactada antes.

  - Resumen estadístico global de 'pdays':
    Media: 51.31 | Desviación Std: 108.78 | Min: -1.0 | Max: 854.0

2. EVALUACIÓN ESTADÍSTICA DE INGENIERÍA DE CARACTERÍSTICAS:
  - P-value de la nueva variable ('contactado_previamente'): 1.92e-130 -> SIGNIFICATIVO

 INSTRUCCIONES FINALES PARA EL PIPELINE DE PREPROCESADO

[COLUMNA 'pdays']: APLICAR EXTRACCIÓN Y TRANSFORMACIÓN AVANZADA.
 -> Motivo: Separar a los clientes 'nunca contactados' (-1) del resto es ALTAMENTE predictivo (p < 0.05).
 -> Acción para el Pipeline: Crear un CustomTransformer que:
    1. Genere la nueva variable binaria 'contactado_previamente' (0 si es -1, 1 si no).
    2. Reemplace el -1 en la columna original de 'pdays' por un valor lógico (ej. la mediana de los sí contactados) para no distorsionar la es

### 2.4.2 Categorías Raras (Rare Labels)

In [ ]:
import pandas as pd
import numpy as np

print("EXTRA: ANÁLISIS MATEMÁTICO DE CATEGORÍAS RARAS (IGNORANDO NULOS/VACÍOS)\n")

# Definimos los umbrales matemáticos
umbral_frecuencia = 0.015  # Consideramos "rara" si aparece en menos del 1.5% de los datos válidos
umbral_diferencia = 0.10   # Diferencia del 10% respecto a la media para considerarla "predictiva"

# Lista de valores que indican datos incompletos o vacíos
valores_incompletos = ['unknown', 'nonexistent', 'other', '?', 'na', 'n/a', 'null', 'none', '-', '', ' ']

# Aseguramos que tenemos el target en formato numérico (0 y 1)
if 'deposit_bin' not in df.columns:
    df['deposit_bin'] = df['deposit'].map({'yes': 1, 'no': 0})

tasa_global = df['deposit_bin'].mean()
print(f"Tasa de conversión global del dataset: {tasa_global*100:.2f}%\n")

for col in categoricas:
    if col == 'deposit': continue

    # 1. FILTRO DE DATOS VÁLIDOS: Ignoramos NaNs y valores de la lista 'valores_incompletos'
    # Pasamos a texto, minúsculas y quitamos espacios para hacer una búsqueda exacta
    col_limpia = df[col].astype(str).str.lower().str.strip()
    mask_validos = ~col_limpia.isin(valores_incompletos) & df[col].notna()

    df_validos = df[mask_validos]

    if len(df_validos) == 0:
        continue # Si toda la columna está vacía (improbable), pasamos a la siguiente

    # 2. ¿Cuáles PODRÍAN ser agrupadas? (Calculamos frecuencia SOLO sobre los válidos)
    frecuencias = df_validos[col].value_counts(normalize=True)
    categorias_raras = frecuencias[frecuencias < umbral_frecuencia].index.tolist()

    if len(categorias_raras) == 0:
        print(f"[{col}]: Ninguna categoría válida es menor al 1.5%. \n  -> DECISIÓN: No agrupar nada.\n")
        continue

    print(f"[{col}]: Entre los datos completos, podrían agruparse {categorias_raras} (frecuencia < 1.5%).")

    categorias_a_agrupar = []
    categorias_a_mantener = []

    # 3. Análisis del impacto predictivo de cada categoría rara
    for cat in categorias_raras:
        # Tasa de éxito de esta categoría específica
        tasa_cat = df_validos[df_validos[col] == cat]['deposit_bin'].mean()
        diferencia = abs(tasa_cat - tasa_global)

        # Lógica IF-ELSE
        if diferencia > umbral_diferencia:
            categorias_a_mantener.append((cat, tasa_cat))
        else:
            categorias_a_agrupar.append((cat, tasa_cat))

    # 4. Veredicto final
    if len(categorias_a_agrupar) > 0:
        nombres_agrupar = [c[0] for c in categorias_a_agrupar]
        print(f"  -> DECISIÓN: SÍ AGRUPAR {nombres_agrupar}.")
        print(f"     ¿Por qué?: Son raras, no predictivas y hemos excluido los nulos del cálculo. Agruparlas en 'otros_agrupado' reducirá el ruido de los modelos.")

    if len(categorias_a_mantener) > 0:
        nombres_mantener = [c[0] for c in categorias_a_mantener]
        print(f"  -> DECISIÓN: NO AGRUPAR {nombres_mantener}.")
        for nombre, tasa in categorias_a_mantener:
            print(f"     ¿Por qué '{nombre}'?: Tasa del {tasa*100:.1f}% vs media global {tasa_global*100:.1f}%. Aporta demasiado valor predictivo como para camuflarla.")
    print("-" * 60)


EXTRA: ANÁLISIS MATEMÁTICO DE CATEGORÍAS RARAS (IGNORANDO NULOS/VACÍOS)
Tasa de conversión global del dataset: 47.45%

[job]: Ninguna categoría válida es menor al 1.5%. 
  -> DECISIÓN: No agrupar nada.

[marital]: Ninguna categoría válida es menor al 1.5%. 
  -> DECISIÓN: No agrupar nada.

[education]: Ninguna categoría válida es menor al 1.5%. 
  -> DECISIÓN: No agrupar nada.

[default]: Ninguna categoría válida es menor al 1.5%. 
  -> DECISIÓN: No agrupar nada.

[housing]: Ninguna categoría válida es menor al 1.5%. 
  -> DECISIÓN: No agrupar nada.

[loan]: Ninguna categoría válida es menor al 1.5%. 
  -> DECISIÓN: No agrupar nada.

[contact]: Ninguna categoría válida es menor al 1.5%. 
  -> DECISIÓN: No agrupar nada.

[month]: Entre los datos completos, podrían agruparse ['dec'] (frecuencia < 1.5%).
  -> DECISIÓN: NO AGRUPAR ['dec'].
     ¿Por qué 'dec'?: Tasa del 91.7% vs media global 47.5%. Aporta demasiado valor predictivo como para camuflarla.
-----------------------------------

Hemos querido analizar la frecuencia de las variables categóricas para ver si habían algunas que podrían ser unificadas en caso de que su aparición fuera marginal para así eliminar ruido del dataset. Al final, esto solo ha ocurrido con 'dec' en la variable *month*. Sin embargo, al descubrir que aportaba demasiado valor predictivo no hemos podido camuflarla.

### 2.4.3 Estacionalidad y Espejismos Temporales

In [16]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

print("ANÁLISIS TEMPORAL UNIFICADO (MES Y DÍA CON DETECCIÓN DE ESPEJISMOS)\n")

# Aseguramos el target numérico si no está
if 'deposit_bin' not in df.columns:
    df['deposit_bin'] = df['deposit'].map({'yes': 1, 'no': 0})

tasa_global = df['deposit_bin'].mean()

# 1. COMPROBACIÓN DE SESGO TEMPORAL MACROECONÓMICO
print("1. ANÁLISIS DE SESGO TEMPORAL (Riesgo macroeconómico):")
if 'year' in df.columns:
    num_years = df['year'].nunique()
    if num_years == 1:
        sesgo_temporal = True
        print(f"  [!] ALERTA: Dataset de un solo año. Riesgo de estacionalidad engañosa.")
    else:
        sesgo_temporal = False
        print(f"  [+] OK: El dataset abarca {num_years} años. Patrones robustos.")
else:
    sesgo_temporal = True
    print("  [?] PRECAUCIÓN: No existe columna 'year'. Asumiremos riesgo de sesgo temporal.")


# 2. DETECCIÓN DE ESPEJISMOS (Volumen vs Éxito)
print("\n2. ANÁLISIS DE ESPEJISMOS DE NEGOCIO:")

# --- Análisis del Mes ---
stats_mes = df.groupby('month').agg(
    volumen=('deposit_bin', 'count'),
    tasa_exito=('deposit_bin', 'mean')
).reset_index()
stats_mes['pct_volumen'] = (stats_mes['volumen'] / len(df)) * 100
mes_estrella = stats_mes.loc[stats_mes['tasa_exito'].idxmax()]

mes_anomalo = None
if mes_estrella['pct_volumen'] < 5.0:
    mes_anomalo = mes_estrella['month']
    print(f"  [!] ESPEJISMO DETECTADO (MES): '{mes_anomalo}' tiene éxito anormal ({mes_estrella['tasa_exito']*100:.1f}%) pero bajísimo volumen ({mes_estrella['pct_volumen']:.1f}%).")
else:
    print(f"  [+] MES ESTRELLA FIABLE: '{mes_estrella['month']}' con {mes_estrella['tasa_exito']*100:.1f}% de éxito y buen volumen ({mes_estrella['pct_volumen']:.1f}%).")

# --- Análisis del Día ---
dia_anomalo = None
if 'day' in df.columns:
    stats_dia = df.groupby('day').agg(
        volumen=('deposit_bin', 'count'),
        tasa_exito=('deposit_bin', 'mean')
    ).reset_index()
    stats_dia['pct_volumen'] = (stats_dia['volumen'] / len(df)) * 100
    dia_estrella = stats_dia.loc[stats_dia['tasa_exito'].idxmax()]

    # Umbral más bajo para el día porque hay 31 posibles valores (Media ~3.2%)
    if dia_estrella['pct_volumen'] < 1.0:
        dia_anomalo = dia_estrella['day']
        print(f"  [!] ESPEJISMO DETECTADO (DÍA): El día '{dia_anomalo}' tiene éxito anormal ({dia_estrella['tasa_exito']*100:.1f}%) pero bajísimo volumen ({dia_estrella['pct_volumen']:.1f}%).")
    else:
        print(f"  [+] DÍA ESTRELLA FIABLE: El día '{dia_estrella['day']}' con {dia_estrella['tasa_exito']*100:.1f}% de éxito y volumen normal ({dia_estrella['pct_volumen']:.1f}%).")


# 3. VALIDACIÓN ESTADÍSTICA PURA (Test Chi-Cuadrado)
print("\n3. VALIDACIÓN ESTADÍSTICA (Test Chi-Cuadrado sobre el resto de datos):")

# A. Para el MES (Aislando el mes anómalo si existe)
df_mes_test = df[df['month'] != mes_anomalo] if mes_anomalo else df
contingency_month = pd.crosstab(df_mes_test['month'], df_mes_test['deposit'])
_, p_value_m, _, _ = chi2_contingency(contingency_month)
estacionalidad_mes_real = p_value_m < 0.05
print(f"  - P-value Meses restantes: {p_value_m:.4f} -> {'SIGNIFICATIVO (Mantener)\n' if estacionalidad_mes_real else 'NO SIGNIFICATIVO (Borrar)\n'}")

# B. Para el DÍA (Aislando el día anómalo si existe)
if 'day' in df.columns:
    df_dia_test = df[df['day'] != dia_anomalo] if dia_anomalo else df
    contingency_day = pd.crosstab(df_dia_test['day'], df_dia_test['deposit'])
    _, p_value_d, _, _ = chi2_contingency(contingency_day)
    estacionalidad_dia_real = p_value_d < 0.05
    print(f"  - P-value Días restantes: {p_value_d:.4f} -> {'SIGNIFICATIVO (Mantener)\n' if estacionalidad_dia_real else 'NO SIGNIFICATIVO (Borrar)\n'}")


# 4. VEREDICTO FINAL PARA EL PIPELINE
print("INSTRUCCIONES FINALES PARA EL PIPELINE DE PREPROCESADO\n")

# Decisiones sobre el mes
if mes_anomalo and estacionalidad_mes_real:
    print(f"[COLUMNA 'month']: MANTENER COLUMNA. Acción previa -> df = df[df['month'] != '{mes_anomalo}']")
elif mes_anomalo and not estacionalidad_mes_real:
    print(f"[COLUMNA 'month']: ELIMINAR COLUMNA (DROP). Todo el mes es ruido temporal sin '{mes_anomalo}'.")
elif not mes_anomalo and estacionalidad_mes_real:
    print(f"[COLUMNA 'month']: MANTENER COLUMNA INTACTA.")
else:
    print(f"[COLUMNA 'month']: ELIMINAR COLUMNA (DROP). No hay estacionalidad mensual.")

# Decisiones sobre el día
if 'day' in df.columns:
    if dia_anomalo and estacionalidad_dia_real:
        print(f"[COLUMNA 'day']: MANTENER COLUMNA. Acción previa -> df = df[df['day'] != {dia_anomalo}]")
    elif dia_anomalo and not estacionalidad_dia_real:
        print(f"[COLUMNA 'day']: ELIMINAR COLUMNA (DROP). Todo el día es ruido temporal sin '{dia_anomalo}'.")
    elif not dia_anomalo and estacionalidad_dia_real:
        print(f"[COLUMNA 'day']: MANTENER COLUMNA INTACTA.")
    else:
        print(f"[COLUMNA 'day']: ELIMINAR COLUMNA (DROP). No hay estacionalidad diaria.")


ANÁLISIS TEMPORAL UNIFICADO (MES Y DÍA CON DETECCIÓN DE ESPEJISMOS)

1. ANÁLISIS DE SESGO TEMPORAL (Riesgo macroeconómico):
  [?] PRECAUCIÓN: No existe columna 'year'. Asumiremos riesgo de sesgo temporal.

2. ANÁLISIS DE ESPEJISMOS DE NEGOCIO:
  [!] ESPEJISMO DETECTADO (MES): 'dec' tiene éxito anormal (91.7%) pero bajísimo volumen (1.0%).
  [+] DÍA ESTRELLA FIABLE: El día '10.0' con 74.4% de éxito y volumen normal (1.5%).

3. VALIDACIÓN ESTADÍSTICA (Test Chi-Cuadrado sobre el resto de datos):
  - P-value Meses restantes: 0.0000 -> SIGNIFICATIVO (Mantener)

  - P-value Días restantes: 0.0000 -> SIGNIFICATIVO (Mantener)

INSTRUCCIONES FINALES PARA EL PIPELINE DE PREPROCESADO

[COLUMNA 'month']: MANTENER COLUMNA. Acción previa -> df = df[df['month'] != 'dec']
[COLUMNA 'day']: MANTENER COLUMNA INTACTA.


Aquí quisimos hacer una valoración de la posible agrupación por estaciones. Sin embargo, al encontrar esa gran variación desmedida respecto al número de llamadas hechas en diciembre, pensamos que podríamos estar ante un espejismo estadístico, lo que llevaría a una eliminación de la variable; al hacer una inspección más profunda del resto de meses, descubrimos que sí son significativos, por lo que solo eliminamos dec como extreme outlier que puede llegar a emperorar nuestras predicciones. Este mismo estudio lo hicimos para los días, pero no encontramos nada relevante.

### 2.4.4 Data Leakage en 'duration'

Hemos descubierto en nuestro EDA una fuga de información en la variable "*duration*". Desde el punto de vista del banco, esta variable no tiene ninguna lógica atenderla, pues la decisión de si llamar o no a un posible cliente no tiene nada que ver con la duración, pues cualquier llamada no tomada tiene duración 0. De esta manera, esto contamina el resultado de los otros datos. Sin embargo, haciendo un pequeño estudio preliminar, indica que si eliminamos la variable *duration* el F1-score sufre un desplome crítico que pone en riesgo los resultados del entrenamiento de nuestro modelo sino la incluimos.

## 2.5 Correlación y Poder Predictivo Global

In [18]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, pointbiserialr

print("EXTRA: MULTICOLINEALIDAD (PARA MODELOS LINEALES)\n")
# Calculamos la correlación de Pearson y buscamos pares altamente correlacionados
corr_matrix = df[numericas].corr().abs()
# Seleccionamos el triángulo superior de la matriz de correlación
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

colineales = [column for column in upper.columns if any(upper[column] > 0.8)]
if len(colineales) > 0:
    print(f"  - ¡Alerta! Alta correlación (>0.8) en: {colineales}")
    print("  -> Eliminar una de las variables del par o aplicar regularización L1 (Lasso).")
else:
    print("  - No se detectó multicolinealidad severa (Pearson < 0.8). Los modelos serán estables.")


print("\nEXTRA: IMPORTANCIA DE ATRIBUTOS")
print("Analizando qué variables tienen mayor poder predictivo sobre 'deposit'...")

print("\n  -- Variables Numéricas (Correlación Punto-Biserial) --")
for col in numericas:
    # Ignoramos pdays aquí porque -1 distorsiona la matemática, lo trataremos aparte
    if col == 'pdays': continue

    corr, p_value = pointbiserialr(df[col], df['deposit_bin'])
    if p_value < 0.05:
        fuerza = "ALTA" if abs(corr) > 0.3 else ("MODERADA" if abs(corr) > 0.1 else "BAJA")
        print(f"  - {col}: Fuerza predictiva (P-value): {fuerza} (r = {corr:.3f})")
    else:
        print(f"  - {col}: NO significativa (p-value >= 0.05). Considerar eliminar para simplificar modelos.")

print("\n  -- Variables Categóricas (Test Chi-Cuadrado) --")
for col in categoricas:
    if col == 'deposit': continue

    contingency_table = pd.crosstab(df[col], df['deposit'])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)

    if p_value < 0.05:
        print(f"  - {col}: Relación significativa con la suscripción (p-value = {p_value:.2e}). Mantener en el modelo.")
    else:
        print(f"  - {col}: NO significativa. La distribución de esta variable es independiente de la suscripción. Podría generar ruido.")


EXTRA: MULTICOLINEALIDAD (PARA MODELOS LINEALES)

  - No se detectó multicolinealidad severa (Pearson < 0.8). Los modelos serán estables.

EXTRA: IMPORTANCIA DE ATRIBUTOS
Analizando qué variables tienen mayor poder predictivo sobre 'deposit'...

  -- Variables Numéricas (Correlación Punto-Biserial) --
  - age: Fuerza predictiva (P-value): BAJA (r = 0.036)
  - balance: Fuerza predictiva (P-value): BAJA (r = 0.082)
  - day: Fuerza predictiva (P-value): BAJA (r = -0.058)
  - duration: Fuerza predictiva (P-value): ALTA (r = 0.452)
  - campaign: Fuerza predictiva (P-value): MODERADA (r = -0.129)
  - previous: Fuerza predictiva (P-value): MODERADA (r = 0.142)

  -- Variables Categóricas (Test Chi-Cuadrado) --
  - job: Relación significativa con la suscripción (p-value = 1.86e-73). Mantener en el modelo.
  - marital: Relación significativa con la suscripción (p-value = 1.81e-23). Mantener en el modelo.
  - education: Relación significativa con la suscripción (p-value = 7.65e-27). Mantener en 